# UniPolfed window round-trip — QREM

This smoke-sized experiment checks the public `polfed`, `patched_polfed`, and `full_polfed` entrypoints against exact diagonalization (ED). A regular finite-window result contains the requested eigenpairs **plus one converged bracket on each side**. Patched-window results contain only their owned interval, so comparisons below use only regular values inside `[lo, hi]`.

In [ ]:
import Pkg
const NOTEBOOK_ROOT = @__DIR__
Pkg.activate(NOTEBOOK_ROOT)

using UniPolfed
using UniPolfed: polfed, patched_polfed, full_polfed, PolfedResult
using LinearAlgebra, SparseArrays, Random, Printf

const RUN_PLOTS = false
const RUN_FULL_SPECTRUM = false  # set true for the longer final check
@assert basename(dirname(pathof(UniPolfed))) == "src"


## QREM fixture

The real-symmetric Hamiltonian is `diag(ε) + hx Σᵢ σᵢˣ`. `L=8` gives a 256-dimensional smoke test that is still large enough to exercise interior windows.

In [ ]:
function qrem_hamiltonian(L::Int; hx::Real=0.7, T=Float64)
    D = 1 << L
    diagonal = randn(T, D)
    rows = Int[]; cols = Int[]; values = T[]
    for row in 1:D
        push!(rows, row); push!(cols, row); push!(values, diagonal[row])
        state = row - 1
        for bit in 0:(L - 1)
            push!(rows, row)
            push!(cols, xor(state, 1 << bit) + 1)
            push!(values, T(hx))
        end
    end
    return sparse(rows, cols, values, D, D)
end

L = 8
Random.seed!(20260714)
H = qrem_hamiltonian(L)
D = size(H, 1)
x0 = randn(D)
exact = sort(eigvals(Symmetric(Matrix(H))))
@printf("L=%d, dimension=%d, spectrum=[%.5f, %.5f]\n", L, D, extrema(exact)...)


## Count target, then a gap-bounded window

The count run defines an interior block. The window edges are placed at the midpoints of the adjacent ED gaps, avoiding ambiguity from floating-point reconstruction at an eigenvalue-valued boundary.

In [ ]:
howmany = 12
count_result = polfed(H, x0, howmany, :middle; overestimate_iters=5.0)
@assert count_result isa PolfedResult
count_vals = sort(collect(count_result.vals))

nearest_exact_index(value) = argmin(abs.(exact .- value))
i0 = nearest_exact_index(first(count_vals))
i1 = nearest_exact_index(last(count_vals))
@assert 1 < i0 <= i1 < D
lo = (exact[i0 - 1] + exact[i0]) / 2
hi = (exact[i1] + exact[i1 + 1]) / 2
ed_in_window = exact[i0:i1]
@printf("count returned %d values; gap window=[%.6f, %.6f]\n", length(count_vals), lo, hi)


In [ ]:
window = (:unrescaled, (lo, hi))
window_result = polfed(H, x0, window; overestimate_iters=5.0)
regular_vals = sort(collect(window_result.vals))
regular_in_window = filter(value -> lo <= value <= hi, regular_vals)

@assert count(<(lo), regular_vals) == 1
@assert count(>(hi), regular_vals) == 1
@printf("regular window returned %d in-window values + %d brackets\n",
        length(regular_in_window), length(regular_vals) - length(regular_in_window))


## Patched entrypoints

The target mode should reproduce the count block. The window mode owns exactly `[lo, hi]`, without exposing regular-window bracket pairs.

In [ ]:
patch_size = 6
patched_target_vals, _, patched_target_report =
    patched_polfed(H, x0, howmany, patch_size, :middle; overestimate_iters=5.0)
patched_window_vals, _, patched_window_report =
    patched_polfed(H, x0, patch_size, window; overestimate_iters=5.0)
patched_target_vals = sort(collect(patched_target_vals))
patched_window_vals = sort(collect(patched_window_vals))

@assert all(value -> lo <= value <= hi, patched_window_vals)
@printf("patched target=%d, patched window=%d\n",
        length(patched_target_vals), length(patched_window_vals))


## Value-by-value checks

Every comparison preserves ascending order. The regular result is filtered to its requested interval before it is compared to patched output or ED.

In [ ]:
function assert_same_values(label, actual, expected; atol=1e-8)
    @assert length(actual) == length(expected) "$label: length mismatch"
    worst = isempty(actual) ? 0.0 : maximum(abs.(actual .- expected))
    @assert worst <= atol "$label: worst error $worst exceeds $atol"
    @printf("✓ %-30s worst |Δ| = %.3e\n", label, worst)
end

assert_same_values("count ↔ ED", count_vals, ed_in_window)
assert_same_values("patched target ↔ count", patched_target_vals, count_vals)
assert_same_values("regular in-window ↔ ED", regular_in_window, ed_in_window)
assert_same_values("patched window ↔ ED", patched_window_vals, ed_in_window)
assert_same_values("regular in-window ↔ patched", regular_in_window, patched_window_vals)


## Optional full-spectrum check and plot

The full-spectrum path is useful but substantially slower than the default smoke path. Enable it at the top when validating changes to patch assembly.

In [ ]:
if RUN_FULL_SPECTRUM
    full_vals, _, full_report = full_polfed(H, x0, 16)
    assert_same_values("full spectrum ↔ ED", sort(collect(full_vals)), exact)
end


In [ ]:
if RUN_PLOTS
    using PythonPlot
    figure(figsize=(9, 3.5))
    plot(exact, zeros(length(exact)), "."; alpha=0.35, label="ED")
    plot(regular_in_window, fill(0.05, length(regular_in_window)), "|"; label="regular in-window")
    plot(filter(value -> value < lo || value > hi, regular_vals), [0.1, 0.1], "x"; label="brackets")
    axvspan(lo, hi; alpha=0.12, color="tab:green")
    yticks([]); xlabel("energy"); legend(); tight_layout(); gcf()
end
